# HEILO Aurora 1.0: ~150 M parâmetros, treinada do zero, 4 horas por dia

**O que é:** a linha média da HEILO, **5× maior** que a Faísca. Usa o motor novo (técnicas do estilo Gemma) e é 100% sua, treinada do zero.

**Como funciona o dia a dia**
1. Abra este notebook, confira `Ambiente de execução → Alterar tipo → GPU` e clique em `Executar tudo`.
2. Ele prepara **uma parte nova** de texto em português (~20–30 min nos primeiros dias), treina até completar ~4 horas e **salva no Drive**.
3. No dia seguinte, `Executar tudo` de novo: ele **continua de onde parou**.
4. Quando chegar a 100%, ele faz sozinho o ajuste de conversa, a avaliação na mesma régua e o envio ao GitHub.

**Quanto tempo:** a meta é ~2 bilhões de tokens. Na **T4**, isso dá umas **2 a 3 semanas** de sessões diárias; se o Colab der uma **L4 ou A100**, fica bem mais rápido. A célula 6 mostra a previsão real a cada dia.

**Dados:** Wikipédia em português (a do seu Drive) + texto da web em português (FineWeb-2, filtrado por qualidade). Nenhum texto gerado por IA de propósito e nenhum peso de outro modelo.

**Antes da 1ª vez:** rode `publicar_heilo.bat` no PC (para o GitHub ter o código da Aurora). O secret `GITHUB_TOKEN` é o mesmo de sempre.

In [ ]:
# 1) Configuração
import time; INICIO = time.time()
REPO = "hugolindo12/HELIO"
BRANCH = "main"
NOME_VERSAO = "aurora-1.0"
MINUTOS_SESSAO = 235        # ~4 h por dia (com folga para salvar)
MAX_PARTES_WEB = 8          # 8 × ~250 M tokens de texto da web
FINALIZAR_AGORA = False     # True = termina hoje mesmo com o que já aprendeu (ajuste + avaliação + GitHub)
import torch, os, sys, json, subprocess
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!"
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print("GPU:", gpu, f"({mem:.0f} GB)")

In [ ]:
# 2) Drive
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_aurora"
PASTA_V1 = "/content/drive/MyDrive/heilo_seed_v1"
os.makedirs(PASTA, exist_ok=True)
assert os.path.exists(f"{PASTA_V1}/tokenizer.json"), "Não achei o tokenizer da HEILO no Drive (pasta heilo_seed_v1)."

In [ ]:
# 3) Código da HEILO (sempre o mais novo: pega também os exemplos que você aprovou)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run("cd /content/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers
from heilo.training import aurora, pretrain, seed_v1
from heilo.models.seed.tokenizer import BPETokenizer
from pathlib import Path
tok = BPETokenizer.from_json(Path(PASTA_V1, "tokenizer.json").read_text(encoding="utf-8"))
L = aurora.lote_para_gpu(gpu, mem)
print("micro-lote", L["micro"], "× acumular", L["acumular"], "| vocabulário", tok.vocab_size)

## Corpus (cresce uma parte por dia)

In [ ]:
# 4) Wikipédia (do Drive) + uma parte nova de texto da web por sessão
wiki = Path(PASTA_V1) / "corpus_grande"
if not (wiki / "meta.json").exists():
    wiki = Path(PASTA) / "wiki"
    if not (wiki / "meta.json").exists():
        pretrain.preparar_corpus(seed_v1.textos_wikipedia(800_000_000), tok, wiki, max_tokens=10**12, log=lambda s: None)
web = Path(PASTA) / "web"
prontas = pretrain.pastas_corpus(web)
if len(prontas) < MAX_PARTES_WEB and not FINALIZAR_AGORA:
    t0 = time.time()
    try:
        aurora.preparar_proxima_parte(web, tok)
    except Exception as e:
        print("não consegui baixar texto da web hoje (continua com o que já tem):", e)
    print(f"preparação: {(time.time()-t0)/60:.0f} min")
# copia para o disco rápido do Colab (ler do Drive direto deixa o treino bem mais lento)
import shutil
CORPUS = []
for p in [wiki] + pretrain.pastas_corpus(web):
    dst = Path("/content/corpus_local") / (p.parent.name + "_" + p.name)
    if not (dst / "meta.json").exists():
        shutil.copytree(p, dst, dirs_exist_ok=True)
    CORPUS.append(dst)
total = sum(json.loads((p / "meta.json").read_text())["tokens_treino"] for p in CORPUS)
print(f"corpus: {len(CORPUS)} parte(s), {total/1e6:,.0f} M tokens")

## Treino do dia

In [ ]:
# 5) Treina até completar a sessão (salva no Drive a cada 250 passos)
cfg = aurora.CONFIG_AURORA
ck = Path(PASTA) / "pretreino.pt"
PASSOS_TOTAL = aurora.TOKENS_ALVO // (32 * cfg.block_size)
passo_inicial = torch.load(ck, map_location="cpu", weights_only=False)["passo"] if ck.exists() else 0
minutos = MINUTOS_SESSAO - (time.time() - INICIO) / 60 - 15
t0 = time.time()
if FINALIZAR_AGORA or minutos < 10:
    minutos = 0.01
res = pretrain.pretreinar(cfg, tok, CORPUS, ck, passos=PASSOS_TOTAL, lote=L["micro"], acumular=L["acumular"],
                          lr=6e-4, lr_min=6e-5, aquecimento=1000, tempo_max_min=minutos,
                          avaliar_cada=250, salvar_cada=250)
modelo = res["modelo"]

In [ ]:
# 6) Progresso e previsão
feitos = res["passo"] - passo_inicial
seg_passo = (time.time() - t0) / max(1, feitos)
prog_arq = Path(PASTA) / "progresso.json"
prog = json.loads(prog_arq.read_text()) if prog_arq.exists() else []
ult = res["hist"][-1] if res["hist"] else {}
if feitos > 0:
    prog.append({"data": time.strftime("%d/%m %H:%M"), "gpu": gpu, "passo": res["passo"], "passos_hoje": feitos,
                 "perda_val": ult.get("perda_val"), "ppl": ult.get("ppl_val"),
                 "tokens_M": round(res["passo"] * 32 * cfg.block_size / 1e6)})
    prog_arq.write_text(json.dumps(prog, indent=1))
for p in prog:
    print(p)
pct = res["passo"] / PASSOS_TOTAL
faltam = (PASSOS_TOTAL - res["passo"]) * seg_passo / 60 / (MINUTOS_SESSAO - 30)
print(f"\nPROGRESSO: {pct:.1%} ({res['passo']:,}/{PASSOS_TOTAL:,} passos) | faltam ~{faltam:.0f} sessão(ões) de 4 h com esta GPU")
modelo.eval()
for inicio in ["O Brasil é", "A capital de Portugal", "Para fazer um bolo"]:
    ids = [tok.doc] + tok.encode(inicio)
    print("→", inicio + tok.decode(modelo.gerar(ids, max_novos=40, temperatura=0.8, parar_em=(tok.fim, tok.doc),
                                                penalidade=1.2, sem_repetir=4)))
TERMINOU = res["passo"] >= PASSOS_TOTAL or FINALIZAR_AGORA
print("\n" + ("TREINO COMPLETO: seguindo para o ajuste de conversa e a avaliação." if TERMINOU
              else "Sessão salva no Drive. Amanhã é só 'Executar tudo' de novo."))

## Final (só roda quando o treino chega a 100% ou com FINALIZAR_AGORA = True)

In [ ]:
# 7) Ajuste de conversa + avaliação + GitHub
if TERMINOU:
    from heilo.training.pipeline import TrainingPipeline
    from heilo.training.ciclos import CiclosSeed
    from heilo.models.seed.gpt import MiniGPTChat
    pipe = TrainingPipeline()
    split = seed_v1.dividir_sft(pipe)
    print(split["manifest"])
    res_sft = pretrain.ajustar_conversa(modelo, split["treino"], split["val"], passos=3000, lote=L["micro"],
                                        lr=4e-5, paciencia=5, pasta_corpus=CORPUS, peso_texto=0.5,
                                        lote_texto=max(2, L["micro"] // 2))
    arq = pretrain.exportar(modelo, Path(PASTA) / "heilo_aurora_1.0.pt",
                            {"versao": NOME_VERSAO, "linha": "Aurora", "passos_pretreino": res["passo"],
                             "progresso": prog, "sft": res_sft})
    ciclos = CiclosSeed(pipe)
    dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                            {"linha": "Aurora", "progresso": prog[-1] if prog else None,
                             "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val")}},
                            origem="HEILO Aurora (~150 M, motor estilo Gemma), do zero em sessões diárias + SFT misturado")
    reg = json.loads(Path("/content/heilo/data/training/versions.json").read_text())
    for v in reg["versoes"]:
        if v["versao"] == NOME_VERSAO:
            v["linha"] = "Aurora"
    Path("/content/heilo/data/training/versions.json").write_text(json.dumps(reg, indent=2, ensure_ascii=False))
    print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
    chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
    for q in ["oi, tudo bem?", "quem é você?", "qual é a capital de portugal?", "o que é uma função em python?",
              "o que faz o g04?", "me fala sobre o Brasil"]:
        print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7,
                                                      penalidade=1.2, sem_repetir=4), "\n")
    add = aurora.preparar_para_git(Path("/content/heilo"))
    subprocess.run(["git", "-C", "/content/heilo", "config", "user.email", "heilo@colab"])
    subprocess.run(["git", "-C", "/content/heilo", "config", "user.name", "HEILO (Colab)"])
    subprocess.run(["git", "-C", "/content/heilo", "add", "-f"] + add + ["data/training/versions.json", "data/training/eval"])
    subprocess.run(["git", "-C", "/content/heilo", "commit", "-q", "-m", f"HEILO Aurora {NOME_VERSAO} (Colab)"])
    r = subprocess.run("cd /content/heilo && git stash -u -q; git pull --rebase && git push", shell=True)
    print("ENVIADO PARA O GITHUB" if r.returncode == 0 else "ERRO no envio: mande um print desta célula")